# NB2 — Small-File Problem & OPTIMIZE + Z-order (lightweight)

Maps to slide §6 Storage Optimization (& Anti-Patterns) + deliverable bullet 2.

> Spark equivalent: `OPTIMIZE delta.\`path\` ZORDER BY (user_id)`
> delta-rs:        `dt.optimize.compact()` + `dt.optimize.z_order(["user_id"])`

**Key idea:** Z-order's value is *file-skipping*. With min/max stats per
file, Delta prunes files whose range can't contain your predicate. That
only works when the table has *multiple files* — if `compact()` collapses
everything to one file, Z-order has nothing to prune.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
import time, random
import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "events_smallfiles")
reset(table_path)  # idempotent

## 1. Manufacture the small-file problem

200 tiny appends → 200 small files. Realistic streaming-ingestion shape.
Each batch is 5K rows × wider schema (≈200 B/row payload) so post-compaction
we still have multiple files with a 256 KB target — required for Z-order
skipping to actually skip something in the benchmark.

In [2]:
random.seed(42)
TARGET_USER = 4242  # the point-query needle

# Pre-built payload pool keeps row generation fast while making each row fat
# enough that compaction doesn't collapse 1M rows into a single 8 MB file.
PAYLOADS = [("p" * 200) + str(i) for i in range(64)]

for batch in range(200):
    rows = pl.DataFrame({
        "event_id":  list(range(batch * 5_000, (batch + 1) * 5_000)),
        "kind":      [random.choice(["click", "view", "scroll", "purchase"]) for _ in range(5_000)],
        # 100K distinct users → before z-order, the target user is scattered
        # across many files; after z-order it clusters into ~1 file.
        "user_id":   [random.randint(1, 100_000) for _ in range(5_000)],
        "payload":   [random.choice(PAYLOADS) for _ in range(5_000)],
    })
    mode = "overwrite" if batch == 0 else "append"
    write_deltalake(table_path, rows.to_arrow(), mode=mode)

dt = DeltaTable(table_path)
files_before = len(dt.file_uris())
print(f"Files before OPTIMIZE: {files_before}")

Files before OPTIMIZE: 200


In [3]:
# [Bằng chứng - chỉ đọc] Trước OPTIMIZE: mỗi file nhỏ cỡ nào, min/max user_id của nó ra sao?
import pyarrow as pa
_rows = pa.table(DeltaTable(table_path).get_add_actions(flatten=True)).to_pylist()
_sizes = [r["size_bytes"] for r in _rows]
files_could_hold_before = sum(1 for r in _rows if r["min.user_id"] <= TARGET_USER <= r["max.user_id"])
print(f"Số file: {len(_rows)}   kích thước mỗi file: {min(_sizes)/1024:.0f}–{max(_sizes)/1024:.0f} KB")
print("Khoảng user_id của 3 file đầu:", [(r["min.user_id"], r["max.user_id"]) for r in _rows[:3]])
print(f"File 'có thể chứa' user_id={TARGET_USER}: {files_could_hold_before}/{len(_rows)}")

Số file: 200   kích thước mỗi file: 64–65 KB
Khoảng user_id của 3 file đầu: [(46, 99993), (7, 99957), (14, 99970)]
File 'có thể chứa' user_id=4242: 200/200


### 📝 Giải thích — small-file problem
- 200 lần `append` → **200 file Parquet** (≥ 100 như yêu cầu) và 200 commit. Mỗi file chỉ ~64 KB (xem cell bằng chứng ở trên).
- Mỗi file chứa 5.000 `user_id` ngẫu nhiên trong 1…100.000, nên min/max của mọi file đều xấp xỉ **[~1, ~100.000]**: các khoảng chồng lấn gần như hoàn toàn. Vì thế với `user_id = 4242`, engine không thể loại file nào dựa vào thống kê — cell trên cho thấy gần như **tất cả** file đều "có thể chứa" user này và phải được mở.

## 2. Benchmark BEFORE optimize

We use delta-rs's native filter pushdown (`to_pyarrow_table(filters=...)`)
which reads per-file `min`/`max` stats from the transaction log and skips
files whose range can't contain the predicate. That's the mechanism we
want to measure — the same one Spark/Trino use.

In [4]:
def bench(label: str, runs: int = 3) -> float:
    """Median of `runs` point-queries using Delta's stats-based file pruning."""
    times = []
    n = 0
    for _ in range(runs):
        dt_local = DeltaTable(table_path)  # fresh metadata read
        t0 = time.perf_counter()
        tbl = dt_local.to_pyarrow_table(
            filters=[("user_id", "=", TARGET_USER), ("kind", "=", "purchase")]
        )
        n = tbl.num_rows
        times.append(time.perf_counter() - t0)
    times.sort()
    median = times[len(times) // 2]
    print(f"{label:25s}  count={n}  median={median*1000:6.1f} ms  (n={runs})")
    return median

before = bench("BEFORE OPTIMIZE")

BEFORE OPTIMIZE            count=5  median=  48.2 ms  (n=3)


## 3. OPTIMIZE (compact small files) + Z-ORDER (co-locate by user_id)

`target_size` is 256 KB so we keep multiple files post-compact — enough
for Z-order's file-skipping to actually skip something.

In [5]:
TARGET_SIZE = 256 * 1024  # 256 KB — keeps ~50 files post-compact for visible pruning

dt = DeltaTable(table_path)
dt.optimize.compact(target_size=TARGET_SIZE)
dt.optimize.z_order(["user_id"], target_size=TARGET_SIZE)

dt = DeltaTable(table_path)  # refresh
files_after = len(dt.file_uris())
print(f"Files after OPTIMIZE+ZORDER: {files_after}  (was {files_before})")

Files after OPTIMIZE+ZORDER: 55  (was 200)


## 4. Benchmark AFTER

In [6]:
after = bench("AFTER OPTIMIZE+ZORDER")
print(f"\nSpeedup: {before/max(after, 1e-6):.1f}×  (target ≥ 3×)")
print(f"File reduction: {files_before} → {files_after}  ({files_before/max(files_after,1):.0f}× fewer)")

AFTER OPTIMIZE+ZORDER      count=5  median=   4.0 ms  (n=3)

Speedup: 11.9×  (target ≥ 3×)
File reduction: 200 → 55  (4× fewer)


### 📝 Giải thích — OPTIMIZE + Z-ORDER và tốc độ
- `compact` gộp file nhỏ; `z_order(["user_id"])` viết lại dữ liệu **sắp xếp theo `user_id`**, mỗi file giữ một dải hẹp. Số file giảm từ **200 xuống 55** (xem output; `target_size=256 KB` được cố ý giữ nhỏ để vẫn còn nhiều file cho Z-order bỏ qua — nếu gộp thành 1 file thì không còn gì để pruning).
- **Speedup** ở trên là thời gian đồng hồ (median của 3 lần chạy) nên **dao động theo máy, cache và tải CPU**; không dùng nó làm bằng chứng duy nhất. Tiêu chí cho phép dùng pruning thay thế, và pruning là con số xác định (xem bên dưới).

## 5. Why this works — inspect file-level stats

Delta stores per-file `min`/`max` for stat-eligible columns in the
transaction log. After Z-order, `user_id` ranges per file are tight and
non-overlapping; the engine prunes files whose range excludes 4242.

In [7]:
import json
import os

log_dir = os.path.join(table_path, "_delta_log")
last_log = sorted(f for f in os.listdir(log_dir) if f.endswith(".json"))[-1]
print(f"Inspecting {last_log}:")
hits = 0
ranges = []
with open(os.path.join(log_dir, last_log)) as fh:
    for line in fh:
        e = json.loads(line)
        if "add" in e and "stats" in e["add"]:
            stats = json.loads(e["add"]["stats"])
            mn = stats.get("minValues", {}).get("user_id")
            mx = stats.get("maxValues", {}).get("user_id")
            if mn is not None:
                ranges.append((mn, mx))
                if mn <= TARGET_USER <= mx:
                    hits += 1
for mn, mx in sorted(ranges):
    marker = " ← contains target" if mn <= TARGET_USER <= mx else ""
    print(f"  file user_id range: [{mn:>6}, {mx:>6}]{marker}")

# Slide-5 deliverable allows EITHER metric — print both so the student can
# pick whichever the grader screenshots:
#   Speedup ≥ 3×              (wall-clock, noisy on local SSD)
#   Files-pruned ratio ≥ 10×  (deterministic, the truer Z-order metric)
pruned_ratio = files_after / max(hits, 1)
print(
    f"\n──── Z-order deliverable metrics ────\n"
    f"  Speedup (wall-clock):   {before/max(after, 1e-6):>5.1f}×   (target ≥ 3×)\n"
    f"  Files-pruned ratio:     {pruned_ratio:>5.1f}×   (target ≥ 10×)   "
    f"[{hits} of {files_after} files cover user_id={TARGET_USER}]"
)

Inspecting 00000000000000000201.json:
  file user_id range: [     1,   1851]
  file user_id range: [  1851,   3696]
  file user_id range: [  3696,   5534] ← contains target
  file user_id range: [  5535,   7366]
  file user_id range: [  7366,   9178]
  file user_id range: [  9178,  11014]
  file user_id range: [ 11014,  12873]
  file user_id range: [ 12874,  14722]
  file user_id range: [ 14722,  16568]
  file user_id range: [ 16568,  18406]
  file user_id range: [ 18406,  20263]
  file user_id range: [ 20263,  22115]
  file user_id range: [ 22115,  23954]
  file user_id range: [ 23954,  25789]
  file user_id range: [ 25790,  27641]
  file user_id range: [ 27641,  29487]
  file user_id range: [ 29487,  31315]
  file user_id range: [ 31315,  33149]
  file user_id range: [ 33149,  34985]
  file user_id range: [ 34985,  36811]
  file user_id range: [ 36811,  38666]
  file user_id range: [ 38666,  40498]
  file user_id range: [ 40498,  42319]
  file user_id range: [ 42319,  44175]
  file u

### 📝 Giải thích — files-pruned ratio và dải min/max
- Sau Z-order, dải `user_id` của các file **nối đuôi nhau, gần như không chồng lấn** (ví dụ `[1, 1851]`, `[1851, 3696]`, `[3696, 5534]`…). Mỗi file chỉ rộng ~1.850 giá trị thay vì ~100.000 như trước.
- Với `user_id = 4242` chỉ **1 file** (`[3696, 5534]`, dòng có `← contains target`) có thể chứa; **54/55 file bị bỏ qua** → **files-pruned ratio = 55 / 1 = 55×** (≥ 10×, đạt tiêu chí độc lập với đồng hồ).
- Cơ chế: Delta ghi min/max từng cột vào log; engine so sánh predicate với khoảng này và không mở file nằm ngoài khoảng. Z-order là thao tác làm cho các khoảng đó **hẹp và rời nhau**, nên thống kê trở nên có ích.

## ✅ Deliverable check
- [ ] Speedup ≥ 3× **or** files-pruned ratio ≥ 10× (slide §6 allows either)
- [ ] File count dropped substantially after compact()
- [ ] Stats inspection shows ~1 file covers `user_id=4242`
- [ ] Screenshot the printed numbers

In [8]:
speedup = before / max(after, 1e-6)
checks = {
    "compaction reduced file count":  files_after < files_before,
    "speedup ≥ 3x OR pruning ≥ 10x":  speedup >= 3 or pruned_ratio >= 10,
    "stats isolate the target user":  hits <= max(2, files_after // 4),
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"\n  (speedup={speedup:.1f}x, pruning={pruned_ratio:.1f}x — the slide accepts EITHER;")
print("   wall-clock is noisy on a laptop, which is why file-pruning is the fallback.)")
assert all(checks.values()), "NB2 incomplete — see FAIL rows above"
print("\nNB2 complete.")

  [PASS] compaction reduced file count
  [PASS] speedup ≥ 3x OR pruning ≥ 10x
  [PASS] stats isolate the target user

  (speedup=11.9x, pruning=55.0x — the slide accepts EITHER;
   wall-clock is noisy on a laptop, which is why file-pruning is the fallback.)

NB2 complete.


### ✅ Tổng kết NB2
| Tiêu chí | Kết quả |
|---|---|
| ≥ 100 file trước OPTIMIZE | 200 file |
| `numFiles` giảm rõ rệt | 200 → 55 |
| Speedup ≥ 3× **hoặc** pruning ≥ 10× | pruning 55× (speedup xem output, dao động theo máy) |